In [5]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib.path import Path
from matplotlib.transforms import Affine2D
from ipywidgets import interact, interactive
import ipywidgets as widgets


#Player Totals Dataset
PD= pd.read_csv("Player Totals.csv").drop(columns=['g','gs','ft','fta','ft_percent','lg'])
PD=PD.query("season >= 1980")

#Team Totals Dataset
TD= pd.read_csv("Team Totals.csv").drop(columns=['lg'])
TD=TD.query("season >= 1980")

#Team Summaries Dataset
TS=pd.read_csv("Team Summaries.csv")

In [2]:
def big_number(ax, value, label, color):
    """Show a large number above a short label in an empty axes."""
    ax.set_axis_off()
    ax.text(0, 0.55, value, fontsize=40, fontweight='bold', color=color,
            transform=ax.transAxes, va='bottom')
    ax.text(0, 0.45, label, fontsize=11, color='dimgray',
            transform=ax.transAxes, va='top')

In [3]:
# Settings and shared tables for the dashboard charts
FIRST, LAST = 1980, 2024
sns.set_theme(style='whitegrid', context='notebook')
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, RED = '#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#e34948'
POS_ORDER = ['PG', 'SG', 'SF', 'PF', 'C']
POS_COLORS = dict(zip(POS_ORDER, [BLUE, ORANGE, AQUA, YELLOW, MAGENTA]))

def true_shooting(pts, fga, fta):
    """TS% = PTS / (2 * (FGA + 0.44 * FTA)), as a percent."""
    return 100 * pts / (2 * (fga + 0.44 * fta))

def decade(season):
    return f"{(season // 10) * 10}s"

# Teams: one row per team per season, with per-game and derived stats
TEAMS = TD.query("@FIRST <= season <= @LAST").copy()
for col in ['pts', 'fga', 'x3pa', 'fta']:
    TEAMS[f'{col}_pg'] = TEAMS[col] / TEAMS['g']
TEAMS['three_share'] = 100 * TEAMS['x3pa'] / TEAMS['fga']
TEAMS['ts'] = true_shooting(TEAMS['pts'], TEAMS['fga'], TEAMS['fta'])

# Basketball-Reference adds a "League Average" row for each season
LEAGUE = TEAMS[TEAMS['team'] == 'League Average'].set_index('season').sort_index()
TEAMS = TEAMS[TEAMS['team'] != 'League Average']

# Champions are not in the dataset, so we list them here
CHAMPIONS = {
    1980: 'LAL', 1981: 'BOS', 1982: 'LAL', 1983: 'PHI', 1984: 'BOS', 1985: 'LAL', 1986: 'BOS',
    1987: 'LAL', 1988: 'LAL', 1989: 'DET', 1990: 'DET', 1991: 'CHI', 1992: 'CHI', 1993: 'CHI',
    1994: 'HOU', 1995: 'HOU', 1996: 'CHI', 1997: 'CHI', 1998: 'CHI', 1999: 'SAS', 2000: 'LAL',
    2001: 'LAL', 2002: 'LAL', 2003: 'SAS', 2004: 'DET', 2005: 'SAS', 2006: 'MIA', 2007: 'SAS',
    2008: 'BOS', 2009: 'LAL', 2010: 'LAL', 2011: 'DAL', 2012: 'MIA', 2013: 'MIA', 2014: 'SAS',
    2015: 'GSW', 2016: 'CLE', 2017: 'GSW', 2018: 'GSW', 2019: 'TOR', 2020: 'LAL', 2021: 'MIL',
    2022: 'GSW', 2023: 'DEN', 2024: 'BOS'}

# Players: keep one row per player per season.
# A traded player has a combined row (team = '2TM', '3TM' or 'TOT') plus one row per team.
PLAYERS = PD.query("@FIRST <= season <= @LAST").dropna(subset=['pos']).copy()
PLAYERS['combined'] = PLAYERS['team'].str.fullmatch(r'\d+TM|TOT')
traded = PLAYERS.loc[PLAYERS['combined'], ['season', 'player_id']].drop_duplicates()
PLAYERS = PLAYERS.merge(traded.assign(was_traded=True), on=['season', 'player_id'], how='left')
PLAYERS = PLAYERS[PLAYERS['combined'] | PLAYERS['was_traded'].isna()]
PLAYERS['pos'] = PLAYERS['pos'].str.split('-').str[0]

STATS_36 = {'x3pa': '3PA', 'trb': 'Rebounds', 'ast': 'Assists'}
for col in STATS_36:
    PLAYERS[f'{col}_36'] = 36 * PLAYERS[col] / PLAYERS['mp']

# Position averages per season, weighted by minutes played
POS_SEASON = PLAYERS.groupby(['season', 'pos'])[['mp', *STATS_36]].sum()
for col in STATS_36:
    POS_SEASON[f'{col}_36'] = 36 * POS_SEASON[col] / POS_SEASON['mp']
POS_SEASON = POS_SEASON.reset_index()

print(f"{len(LEAGUE)} league seasons, {len(TEAMS)} team seasons, {len(PLAYERS)} player seasons")

45 league seasons, 1254 team seasons, 19399 player seasons


In [4]:
def compare_seasons(start=FIRST, end=LAST):
    """Show how four league averages changed between two seasons."""
    if start >= end:
        print('Pick a start season that is before the end season.')
        return
    a, b = LEAGUE.loc[start], LEAGUE.loc[end]
    tiles = [('three_share', 'shots from three', '%'),
             ('ts', 'true shooting', '%'),
             ('pts_pg', 'points per game', ''),
             ('fga_pg', 'FGA per game', '')]
    fig, axes = plt.subplots(1, 4, figsize=(14, 2.2))
    for ax, (col, label, unit) in zip(axes, tiles):
        change = b[col] - a[col]
        big_number(ax, f"{b[col]:.1f}{unit}",
                   f"{label} in {end}\n{change:+.1f} since {start} ({a[col]:.1f}{unit})",
                   BLUE if change >= 0 else ORANGE)
    plt.show()

interact(compare_seasons,
         start=widgets.IntSlider(value=FIRST, min=FIRST, max=LAST - 1, description='Start'),
         end=widgets.IntSlider(value=LAST, min=FIRST + 1, max=LAST, description='End'));

interactive(children=(IntSlider(value=1980, description='Start', max=2023, min=1980), IntSlider(value=2024, de…